In [4]:
!pip install whisper


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import whisper

# Load the base model (you can also use 'small', 'medium', 'large' for better accuracy)
model = whisper.load_model("base")

TypeError: argument of type 'NoneType' is not iterable

In [3]:
!pip install gradio

  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------- ----------- 30.7/43.1 kB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 43.1/43.1 kB 1.0 MB/s eta 0:00:00
  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     -------------- ------------------------ 41.0/109.4 kB 1.9 MB/s eta 0:00:01
     ----------------------------------- -- 102.4/109.4 kB 1.5 MB/s eta 0:00:01
     -------------------------------------- 109.4/109.4 kB 1.3 MB/s eta 0:00:00
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached semantic_version-2.10.0-py2


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import gradio as gr
import whisper
import numpy as np
import tempfile
import os
import soundfile as sf

# Load the Whisper model
model = whisper.load_model("base")

# Function to transcribe audio
def transcribe_audio(audio_path):
    try:
        audio = whisper.load_audio(audio_path)
        audio = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(audio).to(model.device)
        _, probs = model.detect_language(mel)
        detected_language = max(probs, key=probs.get)
        options = whisper.DecodingOptions(fp16=False)
        result = whisper.decode(model, mel, options)
        transcription = result.text
        return transcription, detected_language
    except Exception as e:
        return f"Error in transcription: {str(e)}", "Unknown"

# Function to detect speech disorder (stuttering)
def detect_speech_disorder(transcription):
    words = transcription.split()
    stutter_count = 0
    for i in range(len(words) - 1):
        if words[i] == words[i + 1]:
            stutter_count += 1
    return stutter_count > 0, stutter_count

# Function to analyze articulation issues
def analyze_articulation(transcription, expected_words):
    articulation_issues = []
    for word in expected_words:
        if word not in transcription:
            articulation_issues.append(word)
    return articulation_issues

# Function to process microphone input
def process_microphone(audio):
    try:
        # Save the audio to a temporary file
        sample_rate, audio_data = audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
            sf.write(tmp_file.name, audio_data, sample_rate)
            audio_path = tmp_file.name

        # Transcribe the audio
        transcription, detected_language = transcribe_audio(audio_path)
        os.unlink(audio_path)  # Clean up temporary file

        # Detect speech disorder (stuttering)
        disorder_detected, stutter_count = detect_speech_disorder(transcription)

        # Analyze articulation issues
        expected_words = ["മലയാളം", "ഭാഷ", "സംസാരം", "ശബ്ദം"]  # Add more Malayalam words as needed
        articulation_issues = analyze_articulation(transcription, expected_words)

        # Prepare results
        results = {
            "Transcription": transcription,
            "Detected Language": detected_language,
            "Stuttering Detected": "Yes" if disorder_detected else "No",
            "Stutter Count": stutter_count,
            "Articulation Issues": ", ".join(articulation_issues) if articulation_issues else "None"
        }

        return results
    except Exception as e:
        return {"Error": str(e)}

# Function to process uploaded audio file
def process_uploaded_audio(file):
    try:
        # Transcribe the audio
        transcription, detected_language = transcribe_audio(file)

        # Detect speech disorder (stuttering)
        disorder_detected, stutter_count = detect_speech_disorder(transcription)

        # Analyze articulation issues
        expected_words = ["മലയാളം", "ഭാഷ", "സംസാരം", "ശബ്ദം"]  # Add more Malayalam words as needed
        articulation_issues = analyze_articulation(transcription, expected_words)

        # Prepare results
        results = {
            "Transcription": transcription,
            "Detected Language": detected_language,
            "Stuttering Detected": "Yes" if disorder_detected else "No",
            "Stutter Count": stutter_count,
            "Articulation Issues": ", ".join(articulation_issues) if articulation_issues else "None"
        }

        return results
    except Exception as e:
        return {"Error": str(e)}

# Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# Speech Therapy App for Malayalam 🎤")
    gr.Markdown("Use your microphone or upload an audio file to check for speech disorders and articulation issues.")

    # Real-time microphone input
    with gr.Tab("Microphone Input"):
        audio_input = gr.Audio(sources=["microphone"], type="numpy", label="Record Audio")
        analyze_button = gr.Button("Analyze Microphone Input")

        # Output components for microphone input
        transcription_output_mic = gr.Textbox(label="Transcription")
        language_output_mic = gr.Textbox(label="Detected Language")
        stutter_output_mic = gr.Textbox(label="Stuttering Detected")
        stutter_count_output_mic = gr.Textbox(label="Stutter Count")
        articulation_output_mic = gr.Textbox(label="Articulation Issues")

        # Analyze button action for microphone input
        analyze_button.click(
            fn=process_microphone,
            inputs=audio_input,
            outputs=[transcription_output_mic, language_output_mic, stutter_output_mic, stutter_count_output_mic, articulation_output_mic]
        )

    # Audio file upload
    with gr.Tab("Upload Audio File"):
        file_input = gr.Audio(sources=["upload"], type="filepath", label="Upload Audio File")
        analyze_upload_button = gr.Button("Analyze Uploaded File")

        # Output components for uploaded file
        transcription_output_file = gr.Textbox(label="Transcription")
        language_output_file = gr.Textbox(label="Detected Language")
        stutter_output_file = gr.Textbox(label="Stuttering Detected")
        stutter_count_output_file = gr.Textbox(label="Stutter Count")
        articulation_output_file = gr.Textbox(label="Articulation Issues")

        # Analyze button action for uploaded file
        analyze_upload_button.click(
            fn=process_uploaded_audio,
            inputs=file_input,
            outputs=[transcription_output_file, language_output_file, stutter_output_file, stutter_count_output_file, articulation_output_file]
        )

# Launch the app
demo.launch(share=True)  # Set `share=True` to get a public link

c:\Users\LENOVO\Desktop\MY PROJECTS\MALAYALM SPEECH THEREAPY\MALAYALAM READING FLUENCY ANALYSER\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: argument of type 'NoneType' is not iterable

In [ ]:
#whisper microphone audio testiong
import sounddevice as sd
import numpy as np
import torch

# Load Whisper model
#model = whisper.load_model("base")  # You can use "small", "medium", or "large" for better accuracy

# Function to record audio in real time
def record_audio(duration=5, samplerate=16000):
    print("🎤 Recording... Speak now!")
    audio_data = sd.rec(int(samplerate * duration), samplerate=samplerate, channels=1, dtype=np.float32)
    sd.wait()
    print("✅ Recording finished.")
    return np.squeeze(audio_data)  # Remove extra dimensions

# Function to transcribe live audio
def transcribe_live_audio():
    while True:
        audio = record_audio(duration=5)  # Adjust duration as needed
        mel = whisper.log_mel_spectrogram(audio).to(model.device)
        options = whisper.DecodingOptions(fp16=torch.cuda.is_available())  # Enable FP16 if using GPU
        result = whisper.decode(model, mel, options)
        print("📝 Transcription:", result.text)

# Start live transcription
transcribe_live_audio()

🎤 Recording... Speak now!
✅ Recording finished.


AssertionError: incorrect audio shape

In [5]:
#gradio app without microphone

import gradio as gr
#import whisper

# Load the Whisper model
#model = whisper.load_model("base")

# Function to transcribe audio and detect speech disorder
def transcribe_and_detect(audio_path):
    # Transcribe audio
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    _, probs = model.detect_language(mel)
    detected_language = max(probs, key=probs.get)
    options = whisper.DecodingOptions(fp16=False)
    result = whisper.decode(model, mel, options)
    transcription = result.text

    # Detect speech disorder
    words = transcription.split()
    disorder_detected = False
    for i in range(len(words) - 1):
        if words[i] == words[i + 1]:
            disorder_detected = True
            break
    message = "Stuttering detected." if disorder_detected else "No speech disorder detected."

    # Recommend exercises
    if disorder_detected:
        exercises = [
            "Practice slow reading with a focus on pronunciation.",
            "Use tongue twisters to improve articulation.",
            "Engage in breathing exercises to control speech flow."
        ]
    else:
        exercises = ["No specific exercises recommended. Keep practicing!"]

    return transcription, detected_language, message, exercises

# Gradio interface
interface = gr.Interface(
    fn=transcribe_and_detect,
    inputs=gr.Audio(type="filepath"),  # Use type="filepath" for file upload
    outputs=[
        gr.Textbox(label="Transcription"),
        gr.Textbox(label="Detected Language"),
        gr.Textbox(label="Speech Disorder Detection"),
        gr.Textbox(label="Recommended Exercises")
    ],
    title="Speech Therapy App 🎤",
    description="Upload an audio file to check for speech disorders and get recommended exercises."
)

# Launch the app
interface.launch(share=True)  # Set share=True to get a public link

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://875d38ab7999c0adb1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


c:\Users\LENOVO\Desktop\MY PROJECTS\MALAYALM SPEECH THEREAPY\MALAYALAM READING FLUENCY ANALYSER\myenv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "c:\Users\LENOVO\Desktop\MY PROJECTS\MALAYALM SPEECH THEREAPY\MALAYALAM READING FLUENCY ANALYSER\myenv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\LENOVO\Desktop\MY PROJECTS\MALAYALM SPEECH THEREAPY\MALAYALAM READING FLUENCY ANALYSER\myenv\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\LENOVO\Desktop\MY PROJECTS\MALAYALM SPEECH THEREAPY\MALAYALAM READ

In [ ]:
import gradio as gr
import torchaudio
import torchaudio.transforms as transforms
import os
import json
import random
from datetime import datetime

# User authentication database (in a real app, use a proper database)
USER_DB_FILE = "user_database.json"

# Initialize user database if it doesn't exist
if not os.path.exists(USER_DB_FILE):
    with open(USER_DB_FILE, "w") as f:
        json.dump({}, f)

def load_users():
    with open(USER_DB_FILE, "r") as f:
        return json.load(f)

def save_users(users):
    with open(USER_DB_FILE, "w") as f:
        json.dump(users, f)

# Global session state
session_state = {
    "logged_in": False,
    "username": "",
}

# Authentication functions
def register_user(username, password):
    users = load_users()
    
    if username in users:
        return False, "Username already exists!"
    
    users[username] = {
        "password": password,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "progress": []
    }
    
    save_users(users)
    return True, "Registration successful! Please log in."

def login_user(username, password):
    users = load_users()
    
    if username not in users:
        return False, "Username not found!"
    
    if users[username]["password"] != password:
        return False, "Incorrect password!"
    
    return True, "Login successful!"

# Handle login
def handle_login(username, password):
    success, message = login_user(username, password)
    if success:
        session_state["logged_in"] = True
        session_state["username"] = username
        return message, gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
    return message, gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

# Handle registration
def handle_register(username, password):
    success, message = register_user(username, password)
    if success:
        session_state["logged_in"] = True
        session_state["username"] = username
        return message, gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
    return message, gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

# Handle going to login page from welcome screen
def go_to_login():
    return gr.update(visible=False), gr.update(visible=True)

# Handle logout
def handle_logout():
    session_state["logged_in"] = False
    session_state["username"] = ""
    return "Logged out successfully!", gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

# Function to transcribe and detect speech disorders
def transcribe_and_detect(audio_path):
    if audio_path is None:
        return "No audio detected!", "N/A", "N/A", ["Please record or upload an audio file."]

    # Convert audio to 16 kHz WAV format for Whisper
    try:
        waveform, sample_rate = torchaudio.load(audio_path)
        if sample_rate != 16000:
            resample = transforms.Resample(orig_freq=sample_rate, new_freq=16000)
            waveform = resample(waveform)

        processed_audio_path = "processed_audio.wav"
        torchaudio.save(processed_audio_path, waveform, 16000)
    except Exception as e:
        return f"Audio processing error: {str(e)}", "N/A", "N/A", ["Please try again."]

    # Placeholder for Whisper transcription
    transcription = "ആന അലറലോടലറൽ."

    # Speech disorder detection: Check for repeated words
    words = transcription.split()
    disorder_detected = any(words[i] == words[i + 1] for i in range(len(words) - 1))

    message = "Possible speech disorder detected." if disorder_detected else "No speech disorder detected."

    # Recommended exercises
    exercises = [
        "Practice slow reading with clear pronunciation.",
        "Use breathing exercises to improve speech control.",
        "Repeat tongue twisters to enhance fluency."
    ] if disorder_detected else ["No specific exercises needed. Keep practicing!"]

    return transcription, "Malayalam", message, exercises

# Custom CSS for better styling
custom_css = """
.welcome-container {
    position: relative;
    height: 100vh;
    background-image: url('pexels-a-darmel-7715793.jpg');
    background-size: cover;
    background-position: center;
    display: flex;
    flex-direction: column;
    justify-content: center;
    align-items: center;
    color: white;
    text-align: center;
}

.welcome-title {
    font-family: 'Montserrat', sans-serif;
    font-size: 3.5rem;
    font-weight: 700;
    text-shadow: 2px 2px 8px rgba(0, 0, 0, 0.7);
    margin-bottom: 1rem;
    color: #FF5722;
}

.get-started-btn {
    background-color: #FF5722;
    color: white;
    border: none;
    padding: 8px 16px;
    border-radius: 25px;
    font-size: 0.9rem;
    cursor: pointer;
    transition: all 0.3s ease;
    margin-top: 2rem;
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
    width: 120px !important;
}

.get-started-btn:hover {
    background-color: #E64A19;
    transform: translateY(-2px);
    box-shadow: 0 6px 8px rgba(0, 0, 0, 0.2);
}

.welcome-button-container {
    position: absolute;
    bottom: 20%;
    width: 100%;
}

.blurred-bg {
    background-image: url('file/path/to/your/desktop/image.jpg');
    background-size: cover;
    background-position: center;
    filter: blur(5px);
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    z-index: -1;
}

.content-container {
    background-color: rgba(128, 128, 128, 0.85);
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 8px 16px rgba(0, 0, 0, 0.1);
    position: relative;
    z-index: 1;
    max-width: 800px;
    margin: 20px auto;
    color: white;
}

.content-container h1, .content-container h2, .content-container h3, .content-container h4 {
    color: white;
}

.smaller-text {
    font-size: 0.9rem;
}

.form-container {
    max-width: 400px;
    margin: 0 auto;
}

.footer {
    text-align: center;
    padding: 10px;
    font-size: 0.8rem;
    color: #eee;
    margin-top: 20px;
}

/* Override Gradio default styles */
.dark input, .dark select, .dark textarea {
    background-color: rgba(255, 255, 255, 0.1);
    color: white;
    border-color: rgba(255, 255, 255, 0.2);
}

.tab-nav button {
    color: white !important;
}

.tab-nav button.selected {
    border-color: #FF5722 !important;
    color: white !important;
}
"""

# Create the Gradio app
with gr.Blocks(title="Malayalam Speech Therapy App", css=custom_css) as app:
    # Welcome Page (first screen with background image)
    with gr.Column(visible=True) as welcome_page:
        # Direct button approach without JavaScript
        gr.HTML("""
        <div class="welcome-container">
            <h1 class="welcome-title">Let's Speak with Confidence</h1>
        </div>
        """)
        # Create a regular, visible button centered at the bottom of the screen
        with gr.Row(elem_classes="welcome-button-container"):
            with gr.Column(scale=1):
                gr.HTML("&nbsp;")  # Spacer
            with gr.Column(scale=1):
                welcome_button = gr.Button("Get Started", elem_classes="get-started-btn")
            with gr.Column(scale=1):
                gr.HTML("&nbsp;")  # Spacer

    # Login/Signup Page
    with gr.Column(visible=False) as login_signup_page:
        gr.HTML('<div class="blurred-bg"></div>')
        with gr.Column(elem_classes="content-container"):
            gr.Markdown("# Malayalam Speech Therapy App")
            
            with gr.Tabs() as login_tabs:
                with gr.TabItem("Login") as login_tab:
                    with gr.Column(elem_classes="form-container"):
                        gr.Markdown("### Login to Continue")
                        login_username = gr.Textbox(label="Username", elem_classes="smaller-text")
                        login_password = gr.Textbox(label="Password", type="password", elem_classes="smaller-text")
                        login_button = gr.Button("Login", elem_classes="get-started-btn")
                
                with gr.TabItem("Sign Up") as signup_tab:
                    with gr.Column(elem_classes="form-container"):
                        gr.Markdown("### Sign Up to Get Started")
                        signup_username = gr.Textbox(label="Username", elem_classes="smaller-text")
                        signup_password = gr.Textbox(label="Password", type="password", elem_classes="smaller-text")
                        signup_button = gr.Button("Sign Up", elem_classes="get-started-btn")
            
            gr.HTML('<div class="footer">Malayalam Speech Therapy • Speak with Confidence</div>')
    
    # Intro Page (visible after login/signup)
    with gr.Column(visible=False) as intro_page:
        gr.HTML('<div class="blurred-bg"></div>')
        with gr.Column(elem_classes="content-container"):
            gr.Markdown(f"### Welcome!")
            gr.Markdown("""
            *Welcome to Malayalam Speech Therapy!*
            
            Our app helps you improve your Malayalam pronunciation through targeted exercises and speech analysis.
            
            ### Features:
            """, elem_classes="smaller-text")
            
            gr.HTML("""
            <div class="smaller-text">
                <ul>
                    <li>Pronunciation assessment</li>
                    <li>Customized exercises</li>
                    <li>Progress tracking</li>
                    <li>Focus on difficult Malayalam sounds</li>
                </ul>
            </div>
            """)
            
            get_started_button = gr.Button("Continue", elem_classes="get-started-btn")
            gr.HTML('<div class="footer">Malayalam Speech Therapy • Speak with Confidence</div>')
    
    # Voice Recording/Upload Page
    with gr.Column(visible=False) as voice_page:
        gr.HTML('<div class="blurred-bg"></div>')
        with gr.Column(elem_classes="content-container"):
            gr.Markdown("### Record or Upload Malayalam Speech")
            
            with gr.Row():
                with gr.Column(scale=2):
                    audio_input = gr.Audio(
                        sources=["microphone", "upload"],
                        type="filepath",
                        label="🎤 Record your voice or upload an audio file",
                        elem_classes="smaller-text"
                    )
                
                with gr.Column(scale=1):
                    gr.HTML("""
                    <div style="padding: 10px; border-radius: 10px; background-color: rgba(0, 0, 0, 0.2); margin-top: 16px;">
                        <h4 style="margin-top: 0;">Tips for Recording:</h4>
                        <ul style="padding-left: 20px;" class="smaller-text">
                            <li>Find a quiet environment</li>
                            <li>Speak clearly at a normal pace</li>
                            <li>Position microphone 6-12 inches away</li>
                        </ul>
                    </div>
                    """)
            
            submit_button = gr.Button("Submit", variant="primary", elem_classes="get-started-btn")
            
            gr.Markdown("### Results", elem_classes="smaller-text")
            with gr.Column(elem_classes="smaller-text"):
                transcription_output = gr.Textbox(label="Transcription")
                detected_language = gr.Textbox(label="Detected Language")
                speech_disorder_output = gr.Textbox(label="Speech Disorder Detection")
                recommended_exercises = gr.Textbox(label="Recommended Exercises")
            
            logout_button = gr.Button("Logout", elem_classes="get-started-btn")
            gr.HTML('<div class="footer">Malayalam Speech Therapy • Speak with Confidence</div>')
    
    # Connect event handlers
    welcome_button.click(
        fn=go_to_login,
        inputs=[],
        outputs=[welcome_page, login_signup_page]
    )
    
    login_button.click(
        fn=handle_login,
        inputs=[login_username, login_password],
        outputs=[gr.Textbox(visible=False), intro_page, welcome_page, login_signup_page]
    )
    
    signup_button.click(
        fn=handle_register,
        inputs=[signup_username, signup_password],
        outputs=[gr.Textbox(visible=False), intro_page, welcome_page, login_signup_page]
    )
    
    get_started_button.click(
        fn=lambda: (gr.update(visible=True), gr.update(visible=False)),
        inputs=[],
        outputs=[voice_page, intro_page]
    )
    
    submit_button.click(
        fn=transcribe_and_detect,
        inputs=[audio_input],
        outputs=[transcription_output, detected_language, speech_disorder_output, recommended_exercises]
    )
    
    logout_button.click(
        fn=handle_logout,
        inputs=[],
        outputs=[gr.Textbox(visible=False), intro_page, welcome_page, login_signup_page]
    )

# Launch the app
if __name__ == "__main__":
    app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://eb90fa3795324abbd5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Recording...
Recording finished.


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


transcription:  वोक्रू गरेनु.
mispronounced: ['वोक्रू', 'गरेनु.']
stuttered: []
speed_feedback: Speech speed is normal.
exercises: ['Practice phonetic articulation exercises for: वोक्रू, गरेनु.']
